<a href="https://colab.research.google.com/github/yutaota/intro_to_causal_inference/blob/main/Ex06_IV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. データ生成

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# シードを設定して再現性を確保
np.random.seed(42)

# サンプル数
N = 1000
print("サンプル数：", N)

# 共変量 (X)
X = np.random.randn(N) # 観測できる交絡要因を生成
U = np.random.randn(N) # 観測できない交絡要因を生成

# 操作変数 (Z)
# ZはXに依存して生成（ここでは簡単のため線形関係+ノイズ）
# ZはAに影響を与えるが、YにはXを介さずに直接影響を与えないと仮定
Z = np.random.binomial(1, 0.5, N)
print("当選者数：", sum(Z))

# 処置変数 (A)
# 処置AはZとXに依存して生成（ロジットモデルを使用）
# ZはAに影響を与えるための操作変数としての役割
prob_A = 1 / (1 + np.exp(-(2.0 * Z + 0.4 * X + 0.6 * U + np.random.randn(N) * 0.3)))
A = np.random.binomial(1, prob_A)
print("処置人数：", sum(A))

# アウトカム変数 (Y)
# アウトカムYはAとXに依存して生成
# ZはAを介してのみYに影響を与えると仮定
beta_A = 2.0 # 処置効果の真の値
Y = beta_A * A + 1.5 * X + 1.0 * U + np.random.randn(N) * 1.0
print("処置効果：", beta_A)

# データをDataFrameにまとめる
data = pd.DataFrame({'X': X, 'Z': Z, 'A': A, 'Y': Y})

### 2. 単回帰

In [ ]:
A_ols = sm.add_constant(data[['A']])
y_ols = data['Y']
model_ols = sm.OLS(y_ols, A_ols)
results_ols = model_ols.fit()

print("\n単回帰の結果 (Y ~ A):")
print(results_ols.summary())

## 3. βIV（共変量調整なし）

In [ ]:
# ③ 共変量調整なしの処置効果：Tau
cov_Z_Y = np.cov(data['Z'], data['Y'])[0, 1]
cov_Z_A = np.cov(data['Z'], data['A'])[0, 1]
beta_iv = cov_Z_Y / cov_Z_A
print("beta_iv: ", beta_iv)
print("なんの交絡要因も考慮していないのに真の値に近い値が得られる")

## ２段階最小二乗法（共変量調整あり）

In [ ]:
# ④ 共変量調整ありの処置効果
# --- 2段階推定 ---
# 1段階目: 処置変数 A を 操作変数 Z と 共変量 X で回帰する (線形回帰)
X_stage1 = sm.add_constant(data[['Z', 'X']])
y_stage1 = data['A']
model_stage1 = sm.OLS(y_stage1, X_stage1)
results_stage1 = model_stage1.fit()
data['A_hat'] = results_stage1.predict(X_stage1) # 1段階目の予測値 (Aハット) を取得

# 2段階目: アウトカム Y を 1段階目の予測値 (Aハット) と 共変量 X で回帰する (線形回帰)
X_stage2 = sm.add_constant(data[['A_hat', 'X']])
y_stage2 = data['Y']
model_stage2 = sm.OLS(y_stage2, X_stage2)
results_stage2 = model_stage2.fit()

print("\n2段階目の回帰結果 (Y ~ A_hat + X):")
print(results_stage2.summary())